# Food Detection with YOLO (UEC FOOD 100 subset)

主食・主菜・副菜・汁物の12品目を対象に、YOLOで複数品目の物体検出を行います。
分類演習(`food_CNN_classification.ipynb`)が「1品目だけ切り出した画像」を扱ったのに対し、
こちらは給食トレイのような**1枚の写真に複数品目が写っている**、より実際の場面に近い画像を扱います。

事前準備:
- UEC FOOD 100 データセット一式(1/～100/フォルダ、multiple_food.txt など)をGoogleドライブにアップロード
- `convert_uecfood100_to_yolo.py` と `category_ja_utf8.txt` も同じドライブ上に配置

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip" /content/
!unzip -q /content/dataset100.zip -d /content/
!ls /content/dataset100/1/

[/content/dataset100.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/dataset100.zip or
        /content/dataset100.zip.zip, and cannot find /content/dataset100.zip.ZIP, period.
ls: cannot access '/content/dataset100/1/': No such file or directory


In [ ]:
!ls -la "/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip"

-rw------- 1 root root 991355222 Jul 27 14:41 '/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip'


In [ ]:
!unzip -t "/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip"

unzip:  cannot find or open /content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip, /content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip.zip or /content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip.ZIP.


## 1. Googleドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/food_yolo/'
dataset_root = '/content/UECFOOD100/' # 修正：データセットが展開された正しいパスに設定
output_dir = base_dir + 'yolo_data/'
category_names_path = base_dir + 'category_ja_utf8.txt'
convert_script_path = base_dir + 'convert_uecfood100_to_yolo.py'

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/food_yolo/UECFOOD100/"

1    17  25  33  41  5	 58  66  74  82  90  99
10   18  26  34  42  50  59  67  75  83  91  category_ja_euc.txt
100  19  27  35  43  51  6   68  76  84  92  category_ja_sjis.txt
11   2	 28  36  44  52  60  69  77  85  93  category_ja_utf8.txt
12   20  29  37  45  53  61  7	 78  86  94  category.txt
13   21  3   38  46  54  62  70  79  87  95  multiple_food.txt
14   22  30  39  47  55  63  71  8   88  96  README.txt
15   23  31  4	 48  56  64  72  80  89  97
16   24  32  40  49  57  65  73  81  9	 98


In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/food_yolo/UECFOOD100/1/"

10572.jpg  11590.jpg  13646.jpg  13888.jpg  14206.jpg  14616.jpg  14923.jpg
10586.jpg  11593.jpg  13671.jpg  13893.jpg  14235.jpg  14619.jpg  14987.jpg
10617.jpg  11597.jpg  13676.jpg  13898.jpg  14237.jpg  14621.jpg  14992.jpg
10618.jpg  11608.jpg  13690.jpg  13903.jpg  14239.jpg  14626.jpg  14.jpg
10621.jpg  11649.jpg  13693.jpg  13933.jpg  14240.jpg  14636.jpg  15019.jpg
10628.jpg  11650.jpg  13695.jpg  13938.jpg  14249.jpg  14642.jpg  15041.jpg
10630.jpg  11686.jpg  13696.jpg  13947.jpg  14263.jpg  14644.jpg  15083.jpg
10633.jpg  11689.jpg  13701.jpg  13948.jpg  14267.jpg  14651.jpg  15085.jpg
10634.jpg  11694.jpg  13702.jpg  13958.jpg  14287.jpg  14663.jpg  15098.jpg
10640.jpg  11732.jpg  13704.jpg  13960.jpg  14289.jpg  14666.jpg  15104.jpg
10645.jpg  11737.jpg  13705.jpg  13969.jpg  14291.jpg  14668.jpg  15113.jpg
10666.jpg  11774.jpg  13708.jpg  13978.jpg  14309.jpg  14689.jpg  15134.jpg
10740.jpg  11862.jpg  13723.jpg  13985.jpg  14329.jpg  14698.jpg  15148.jpg
10753.jpg  1195

## 2. ライブラリのインストール

In [ ]:
!pip install ultralytics pillow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.3 MB/s eta 0:00:00


## 3. データ変換(bb_info.txt → YOLO形式)

以前用意した `convert_uecfood100_to_yolo.py` を、選んだ12品目を指定して実行します。
同じ写真が複数カテゴリに重複している場合(例: ごはん+味噌汁)も、自動的に1枚の画像・
1つのラベルファイルにまとめられます。

In [ ]:
CATEGORIES = "1,36,46,55,56,60,63,67,69,70,87,90"

import os

# convert_script_path が存在するか確認
if not os.path.exists(convert_script_path):
    print(f"エラー: スクリプトファイル '{convert_script_path}' が見つかりません。Googleドライブにアップロードされているか確認してください。")
else:
    # コマンド文字列をPythonで明示的に構築し、変数が正しく展開されるようにする
    command = f"python \"{convert_script_path}\" " \
              f"--dataset-root \"{dataset_root}\" " \
              f"--output-dir \"{output_dir}\" " \
              f"--categories \"{CATEGORIES}\" " \
              f"--category-names \"{category_names_path}\" " \
              f"--val-fraction 0.2 " \
              f"--seed 42"
    !{command}

[warn] /content/UECFOOD100/multiple_food.txt not found; treating all images as single-category
[warn] missing /content/UECFOOD100/1/bb_info.txt, skipping category 1
[warn] missing /content/UECFOOD100/36/bb_info.txt, skipping category 36
[warn] missing /content/UECFOOD100/46/bb_info.txt, skipping category 46
[warn] missing /content/UECFOOD100/55/bb_info.txt, skipping category 55
[warn] missing /content/UECFOOD100/56/bb_info.txt, skipping category 56
[warn] missing /content/UECFOOD100/60/bb_info.txt, skipping category 60
[warn] missing /content/UECFOOD100/63/bb_info.txt, skipping category 63
[warn] missing /content/UECFOOD100/67/bb_info.txt, skipping category 67
[warn] missing /content/UECFOOD100/69/bb_info.txt, skipping category 69
[warn] missing /content/UECFOOD100/70/bb_info.txt, skipping category 70
[warn] missing /content/UECFOOD100/87/bb_info.txt, skipping category 87
[warn] missing /content/UECFOOD100/90/bb_info.txt, skipping category 90
Categories processed: 0/12
Images written: 

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip" /content/
!unzip -q /content/dataset100.zip -d /content/
!ls /content/UECFOOD100/1/ | grep bb_info

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/food_yolo/dataset100.zip': No such file or directory
[/content/dataset100.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/dataset100.zip or
        /content/dataset100.zip.zip, and cannot find /content/dataset100.zip.ZIP, period.
ls: cannot access '/content/UECFOOD100/1/': No such file or directory


In [ ]:
!wget -q http://foodcam.mobi/dataset100.zip -O /content/dataset100.zip
!unzip -q /content/dataset100.zip -d /content/
!ls /content/UECFOOD100/1/ | grep bb_info

bb_info.txt


In [ ]:
# 出力内容の確認
import os

for split in ("train", "val"):
    # フォルダが存在しない場合にエラーにならないよう、ディレクトリをチェック
    images_path = output_dir + f"images/{split}"
    labels_path = output_dir + f"labels/{split}"

    n_images = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
    n_labels = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0

    print(f"{split}: images={n_images}, labels={n_labels}")

print()
# output_dirにdata.yamlが存在するかチェック
data_yaml_path = output_dir + "data.yaml"
if os.path.exists(data_yaml_path):
    with open(data_yaml_path, encoding="utf-8") as f:
        print(f.read())
else:
    print(f"エラー: data.yaml が '{data_yaml_path}' に見つかりません。")

train: images=1814, labels=1814
val: images=453, labels=453

path: /content/drive/MyDrive/Colab Notebooks/food_yolo/yolo_data
train: images/train
val: images/val
nc: 12
names:
  0: ごはん
  1: 味噌汁
  2: 鮭の塩焼
  3: 鶏の唐揚げ
  4: 豚カツ
  5: ハンバーグ
  6: 豚肉の生姜焼き
  7: 卵焼き
  8: 納豆
  9: 冷奴
  10: グリーンサラダ
  11: 豚汁



## 4. 学習(Fine-tuning)

In [ ]:
from ultralytics import YOLO

# nano(最小・最速)モデルをベースに、今回の12クラスでfine-tuning
model = YOLO("yolo11n.pt")

results = model.train(
    data=output_dir + "data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="food_yolo",
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/food_yolo/yolo_data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, ex

## 5. 評価(mAP・Precision・Recallなど)

Roboflow側の結果(mAP@50, Precision, Recall, F1)と同じ指標で比較できるよう、検証データで評価します。

In [ ]:
metrics = model.val()

print(f"mAP@50:    {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,584,492 parameters, 0 gradients, 6.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.3 ms, read: 34.9±26.8 MB/s, size: 67.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Colab Notebooks/food_yolo/yolo_data/labels/val.cache... 453 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 453/453 146.2Mit/s 0.0s
val: /content/drive/MyDrive/Colab Notebooks/food_yolo/yolo_data/images/val/multi_86.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 4.2it/s 6.9s
                   all        453        588      0.771      0.749      0.792      0.551
                   ごはん        125        126      0.861      0.937       0.93      0.6

## 6. 推論と可視化

学習したモデルで、実際の写真から複数品目を検出してみます。
Roboflow側で作った `detect_and_show()` と同じ使い勝手になるよう、
同名の関数として揃えています(こちらは `detect_and_show_yolo` という名前です)。

In [ ]:
import matplotlib.pyplot as plt

def detect_and_show_yolo(image_path, confidence=0.4):
    result = model.predict(image_path, conf=confidence, verbose=False)[0]

    annotated = result.plot()  # ultralyticsが描画済みの画像(BGR)を返す
    annotated_rgb = annotated[..., ::-1]  # BGR -> RGB

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(annotated_rgb)
    ax.axis('off')
    plt.show()
    return result

# 使い方
result = detect_and_show_yolo("test_photo.jpg")

## 7. 2つのモデルを見比べる

同じ写真に対して、Roboflow側のモデル(`detect_and_show`)と、
今回のYOLOモデル(`detect_and_show_yolo`)を並べて実行すれば、
「少数の自分の写真だけで学習したモデル」と「既存の大きいデータセットで学習したモデル」の
違いを直接見比べられます。Roboflow側のセルをこのノートブックにコピーしてから、
以下のように両方を呼び出してみてください。

```python
print("--- Roboflow (自分の写真のみで学習) ---")
detect_and_show("test_photo.jpg")

print("--- YOLO (既存の大きいデータセットで学習) ---")
detect_and_show_yolo("test_photo.jpg")
```